# PSA Translation — mT5-small Fine-Tuning
### English/Kiswahili → Ekegusii (low-resource) machine translation

This notebook trains and evaluates **mT5-small** with layer freezing on a curated
English/Kiswahili → Ekegusii PSA (Public Service Announcement) dataset.

**No Colab, Kaggle, or Weights & Biases dependency** — designed to run on any standard
Python + GPU environment (e.g. Navon Cloud JupyterLab). All logs, metrics, and checkpoints
are written to local files under `./checkpoints/` and `./logs/`.

**Requirements:** Python 3.10+, a CUDA GPU (tested on NVIDIA T4 15GB; will run faster/larger
batches on an A100), and `Final_merged_psas.csv` placed in the same directory as this notebook
(or update `DATA_PATH` below).


## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate


In [ ]:
import os
# Restrict to a single GPU by default -- safe no-op on single-GPU machines,
# and avoids a known multi-GPU memory-overhead issue on some setups.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")


In [ ]:
import os, json, time, random
import numpy as np
import pandas as pd
import torch

from transformers import set_seed

# Reproducibility
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Local output directories (created automatically, no cloud mounts needed)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)


## 2. Load curated dataset

In [ ]:
# Place Final_merged_psas.csv in the same folder as this notebook,
# or set DATA_PATH to its full path.
DATA_PATH = "Final_merged_psas.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()


### 2.1 Build the combined (English + Kiswahili) → Ekegusii dataset

Each row becomes **two** training examples where possible: one with English as source,
one with Kiswahili as source, both mapping to the same Ekegusii target.

In [ ]:
def build_combined(df):
    rows = []
    for _, r in df.iterrows():
        if pd.notna(r["English"]) and str(r["English"]).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["English"], "target_text": r["Ekegusii"],
                "source_lang": "en"
            })
        if pd.notna(r.get("Kiswahili")) and str(r.get("Kiswahili")).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["Kiswahili"], "target_text": r["Ekegusii"],
                "source_lang": "sw"
            })
    return pd.DataFrame(rows).dropna(subset=["target_text"])

combined = build_combined(df)
combined = combined[combined["target_text"].astype(str).str.strip() != ""]
print("Total combined examples:", len(combined))
print(combined["source_lang"].value_counts())
print(combined["Domain"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

# stratify by source_lang so both directions are represented in every split
train_df, temp_df = train_test_split(combined, test_size=0.2, random_state=42,
                                      stratify=combined["source_lang"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42,
                                    stratify=temp_df["source_lang"])

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
train_df["source_lang"].value_counts(), test_df["source_lang"].value_counts()


**Low-resource note:** Ekegusii is not in mT5's pretraining language list,
so this is a genuine low-resource target. English is well covered by mT5's pretraining;
Kiswahili is present but underrepresented relative to English.

## 3. Shared utilities: tokenization, metrics, layer freezing, timing

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import evaluate

sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def to_hf(d):
    return Dataset.from_pandas(d[["source_text", "target_text", "source_lang", "Domain"]]
                                .reset_index(drop=True))

train_ds = to_hf(train_df)
val_ds   = to_hf(val_df)
test_ds  = to_hf(test_df)

def build_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        # -100 is the label-ignore sentinel; must be swapped for a real pad id before decoding
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        bleu = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        c = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        return {"bleu": bleu["score"], "chrf": c["score"]}
    return compute_metrics

def freeze_encoder_layers(model, num_layers_to_freeze):
    """Freeze bottom N encoder layers to reduce overfitting risk on our small,
    low-resource fine-tuning set and cut compute cost."""
    encoder = model.get_encoder()
    layers = encoder.block if hasattr(encoder, "block") else encoder.layers
    for i, layer in enumerate(layers):
        if i < num_layers_to_freeze:
            for p in layer.parameters():
                p.requires_grad = False
    return model

def score(preds, refs, label):
    """Computes BLEU/chrF, prints, and returns a result dict (no external logging service)."""
    bleu = sacrebleu.compute(predictions=preds, references=[[r] for r in refs])
    c = chrf.compute(predictions=preds, references=[[r] for r in refs])
    print(f"{label:35s} BLEU={bleu['score']:.2f}  chrF={c['score']:.2f}")
    return {"name": label, "bleu": bleu["score"], "chrf": c["score"]}

MAX_LEN = 128
results_log = []          # collects every score() call for the final summary table
timing_log = {}           # collects wall-clock training time


## 4. mT5-small — tokenizer and preprocessing

In [ ]:
MT5_CHECKPOINT = "google/mt5-small"
mt5_tok = AutoTokenizer.from_pretrained(MT5_CHECKPOINT)

def mt5_prefix(source_lang):
    return "translate English to Ekegusii: " if source_lang == "en" else "translate Kiswahili to Ekegusii: "

def preprocess_mt5(batch):
    inputs = [mt5_prefix(sl) + t for sl, t in zip(batch["source_lang"], batch["source_text"])]
    model_inputs = mt5_tok(inputs, max_length=MAX_LEN, truncation=True)
    labels = mt5_tok(text_target=batch["target_text"], max_length=MAX_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok_mt5 = train_ds.map(preprocess_mt5, batched=True)
val_tok_mt5   = val_ds.map(preprocess_mt5, batched=True)


### 4.1 Baseline (zero-shot) — mT5, before any fine-tuning

In [ ]:
base_mt5 = AutoModelForSeq2SeqLM.from_pretrained(MT5_CHECKPOINT)
base_mt5.to("cuda" if torch.cuda.is_available() else "cpu")

def generate_mt5(model, texts, source_langs, max_new_tokens=MAX_LEN, batch_size=16):
    inputs = [mt5_prefix(sl) + t for sl, t in zip(source_langs, texts)]
    all_preds = []
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i + batch_size]
        enc = mt5_tok(batch, return_tensors="pt", padding=True, truncation=True,
                      max_length=MAX_LEN).to(model.device)
        out = model.generate(**enc, max_length=max_new_tokens)
        all_preds.extend(mt5_tok.batch_decode(out, skip_special_tokens=True))
        del enc, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return all_preds

# Evaluate baseline separately for each source language (needed for domain/ablation tables)
test_en = test_df[test_df["source_lang"] == "en"]
test_sw = test_df[test_df["source_lang"] == "sw"]

preds_base_en = generate_mt5(base_mt5, list(test_en["source_text"]), list(test_en["source_lang"]))
results_log.append(score(preds_base_en, list(test_en["target_text"]), "mt5_zero-shot_en-guz"))

preds_base_sw = generate_mt5(base_mt5, list(test_sw["source_text"]), list(test_sw["source_lang"]))
results_log.append(score(preds_base_sw, list(test_sw["target_text"]), "mt5_zero-shot_sw-guz"))

del base_mt5
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### 4.2 Fine-tuning — mT5 (few-shot), with layer freezing

Checkpoints save automatically each epoch to `checkpoints/mt5_combined_guz/`.
Training logs (loss, BLEU, chrF per epoch) are written to `logs/mt5_training_log.csv`
after training completes — no external logging service required.

In [ ]:
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MT5_CHECKPOINT)
mt5_model = freeze_encoder_layers(mt5_model, num_layers_to_freeze=4)  # mt5-small: 8 encoder layers

data_collator_mt5 = DataCollatorForSeq2Seq(mt5_tok, model=mt5_model)

args_mt5 = Seq2SeqTrainingArguments(
    output_dir="checkpoints/mt5_combined_guz",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    num_train_epochs=5,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    report_to="none",            # no external logging service
    fp16=False,                  # mT5 is unstable in fp16 on T4-class GPUs -- train in fp32
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    gradient_checkpointing=True,
)

trainer_mt5 = Seq2SeqTrainer(
    model=mt5_model,
    args=args_mt5,
    train_dataset=train_tok_mt5,
    eval_dataset=val_tok_mt5,
    data_collator=data_collator_mt5,
    processing_class=mt5_tok,
    compute_metrics=build_compute_metrics(mt5_tok),
)

t0 = time.time()
trainer_mt5.train()
timing_log["mt5_train_seconds"] = time.time() - t0
print(f"mT5 training time: {timing_log['mt5_train_seconds']/60:.1f} minutes")

# Save the best checkpoint (load_best_model_at_end=True already loaded it into memory) as a
# clean, inference-ready model -- this is what the demo function below loads.
trainer_mt5.save_model("checkpoints/mt5_combined_guz/final")
mt5_tok.save_pretrained("checkpoints/mt5_combined_guz/final")

# --- Save training log locally (replaces the W&B dashboard) ---
log_df = pd.DataFrame(trainer_mt5.state.log_history)
log_df.to_csv("logs/mt5_training_log.csv", index=False)
print("Training log saved to logs/mt5_training_log.csv")


### 4.3 Fine-tuned evaluation — mT5, per source language

In [ ]:
preds_ft_en = generate_mt5(trainer_mt5.model, list(test_en["source_text"]), list(test_en["source_lang"]))
results_log.append(score(preds_ft_en, list(test_en["target_text"]), "mt5_few-shot_en-guz"))

preds_ft_sw = generate_mt5(trainer_mt5.model, list(test_sw["source_text"]), list(test_sw["source_lang"]))
results_log.append(score(preds_ft_sw, list(test_sw["target_text"]), "mt5_few-shot_sw-guz"))

# Persist all results (zero-shot + few-shot) locally
with open("logs/mt5_results.json", "w") as f:
    json.dump(results_log, f, indent=2)
print("Results saved to logs/mt5_results.json")


## 5. Hyperparameters, training time, and results

Auto-generated from what actually ran, and saved to `logs/mt5_hyperparameters.csv`
and `logs/mt5_results_table.csv` for the write-up.

In [ ]:
hyperparam_table = pd.DataFrame([{
    "Model": "mT5-small",
    "Pair": "combined (en+sw)->guz",
    "Epochs": args_mt5.num_train_epochs,
    "Batch size": args_mt5.per_device_train_batch_size,
    "Learning rate": args_mt5.learning_rate,
    "Frozen encoder layers": "4/8",
    "fp16": args_mt5.fp16,
    "Gradient checkpointing": args_mt5.gradient_checkpointing,
    "Train time (min)": round(timing_log.get("mt5_train_seconds", 0) / 60, 1),
}])
hyperparam_table.to_csv("logs/mt5_hyperparameters.csv", index=False)
hyperparam_table


In [ ]:
results_df = pd.DataFrame(results_log)
results_df.to_csv("logs/mt5_results_table.csv", index=False)
print("=== Zero-shot vs Few-shot results ===")
print(results_df.to_string(index=False))


## 6. Inference demo

Loads the fine-tuned model from `checkpoints/mt5_combined_guz/final` and translates
sample sentences. Run this cell independently (after training, or in a fresh session
that has skipped straight to this section) to demonstrate translation on new input.

In [ ]:
FINAL_MODEL_DIR = "checkpoints/mt5_combined_guz/final"

# If this cell is run standalone (e.g. a fresh kernel after training already happened),
# reload the fine-tuned model + tokenizer from disk instead of relying on in-memory objects.
if "trainer_mt5" not in globals():
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    mt5_tok = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
    _mt5_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_MODEL_DIR)
    _mt5_model.to("cuda" if torch.cuda.is_available() else "cpu")
else:
    _mt5_model = trainer_mt5.model

def mt5_prefix(source_lang):
    return "translate English to Ekegusii: " if source_lang == "en" else "translate Kiswahili to Ekegusii: "

def translate_psa(text, source_lang="en"):
    """
    text: input sentence (English or Kiswahili)
    source_lang: 'en' (English) or 'sw' (Kiswahili)
    Returns: Ekegusii translation from the fine-tuned mT5 model
    """
    inputs = mt5_tok(mt5_prefix(source_lang) + text, return_tensors="pt",
                      truncation=True, max_length=MAX_LEN).to(_mt5_model.device)
    out = _mt5_model.generate(**inputs, max_length=MAX_LEN)
    return mt5_tok.decode(out[0], skip_special_tokens=True)

# --- Demo: sample PSAs ---
samples = [
    ("Farmers are urged to prioritize safe agrochemical usage this season.", "en"),
    ("Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.", "sw"),
]

for text, lang in samples:
    print(f"[{lang}] {text}")
    print("  mT5 ->", translate_psa(text, source_lang=lang))
    print()
